# 1. Project Introduction

### Project Overview
This project is centered on the **NGA-West2 (Next Generation Attenuation for Western United States)** earthquake ground-motion database. The NGA-West2 flatfile, compiled by the Pacific Earthquake Engineering Research Center (PEER), contains worldwide shallow crustal earthquake records used extensively by earthquake engineers and seismologists to develop empirical ground-motion models (GMMs).

Our project draws inspiration from the Stanford University study by Alan Poulos and Rodrigo Silva-Lopez (*"Machine Learning Tools for Predicting Ground Motion Directionality"*). In that study, the authors explore how ground-shaking orientation varies across geographic space and attempt to predict the absolute difference between orientations of maximum shaking ($\Delta\theta = |\theta_i - \theta_j|$) between pairs of recording stations.

### Role of This Notebook (Phase 1: Data & Feature Engineering)
This notebook is dedicated entirely to the **Data Inspection, Cleaning, and Feature Preparation** phase of the project:
1. **Data Inspection**: Systematically load and understand the 274 attributes and 21,540 ground-motion records in the flatfile.
2. **Quality & Anomaly Diagnostics**: Detect hidden sentinel missing values (`-999`), constant columns, duplicate records, and unused columns.
3. **Seismological Feature Selection**: Identify physically meaningful attributes covering earthquake source properties, site soil conditions, propagation distances, and ground-motion intensities.
4. **Data Cleaning**: Filter out non-significant records using defensible engineering criteria (e.g., minimum magnitude threshold of $M_w \ge 5.0$ as established in the literature) and handle missing records without corrupting the dataset.
5. **Feature Preparation**: Identify categorical features for encoding and separate raw features from training-set-dependent scaling to avoid data leakage.
6. **Dataset Export**: Export a pristine, ML-ready dataset (`cleaned_dataset_NGA_West2_d050.csv`) as variable `cleaned_df` for team members to use in subsequent modeling tasks.

### Important Boundary Conditions
- **No Model Training in this Notebook**: Regression and Neural Network model training are reserved for subsequent phases.
- **No Artificial $\Delta\theta$ Generation**: The paper’s orientation of maximum shaking ($\theta$) requires processing raw two-component accelerograms (RotD100). The supplied RotD50 flatfile does not contain those waveforms. This notebook retains `Theta.D (deg)` and `Source to Site Azimuth (deg)` as flatfile geometry fields so the team can reproduce the current downstream proxy analysis, but `Theta.D` is not the paper’s maximum-shaking orientation. Any pairwise difference based on `Theta.D` must be labeled as a proxy and must not be compared as the paper’s true $\Delta\theta$. This notebook itself does not calculate station pairs or $\Delta\theta$.

# 2. Import Required Libraries

We import standard, beginner-friendly Python data science libraries.
- `pandas` for tabular data manipulation and cleaning.
- `numpy` for numerical array operations.
- `matplotlib.pyplot` for creating figures and plots.
- `seaborn` for statistical visualizations.
- `openpyxl` for inspecting Excel workbook structures.

In [ ]:
#pip install notebook ipykernel pandas numpy matplotlib seaborn openpyxl scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

All required libraries have been loaded successfully.

# 3. Load the Dataset

We locate the raw Excel file `Updated_NGA_West2_Flatfile_RotD50_d050_public_version.xlsx` in the project folder. First, we inspect the available sheets without loading the entire file into memory. Then, we load the active sheet into a pandas DataFrame named `raw_df`.

In [ ]:
excel_path = "dataset/Updated_NGA_West2_Flatfile_RotD50_d050_public_version.xlsx"
wb = openpyxl.load_workbook(excel_path, read_only=True)
available_sheets = wb.sheetnames
print("Available Sheet Names:", available_sheets)
selected_sheet = available_sheets[0]
print("Selected Sheet for Analysis:", selected_sheet)
wb.close()

Now we load the selected sheet into pandas and check its dimensions.

In [ ]:
raw_df = pd.read_excel(excel_path, sheet_name=selected_sheet)
print("DataFrame Shape (Rows, Columns):", raw_df.shape)
print("Total Number of Rows:", raw_df.shape[0])
print("Total Number of Columns:", raw_df.shape[1])

Displaying the first 5 records of the raw dataset:

In [ ]:
raw_df.head()

Displaying the last 5 records of the raw dataset:

In [ ]:
raw_df.tail()

# 4. Understand the Dataset

Here we examine the column types, count numerical versus categorical features, compute descriptive statistics for prominent seismic variables, and construct a column-level summary table.

In [ ]:
numerical_columns = raw_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = raw_df.select_dtypes(include=["object"]).columns.tolist()

print("Number of Numerical Columns:", len(numerical_columns))
print("Number of Categorical/Object Columns:", len(categorical_columns))

Creating a summary table of dataset columns showing data types, missing value count (as initially recognized by pandas), and unique value counts for the first 30 columns:

In [ ]:
column_summary_df = pd.DataFrame({
    "Column Name": raw_df.columns,
    "Data Type": raw_df.dtypes.values,
    "Standard Missing (NaN)": raw_df.isna().sum().values,
    "Unique Values Count": raw_df.nunique().values
})

column_summary_df.head(30)

Computing descriptive statistics for key earthquake, site, distance, and ground motion variables:

In [ ]:
primary_cols = [
    "Earthquake Magnitude",
    "Hypocenter Depth (km)",
    "Vs30 (m/s) selected for analysis",
    "Joyner-Boore Dist. (km)",
    "Campbell R Dist. (km)",
    "ClstD (km)",
    "PGA (g)",
    "PGV (cm/sec)",
    "PGD (cm)"
]

raw_df[primary_cols].describe()

Notice in the descriptive table above that the minimum value for `Earthquake Magnitude`, `Hypocenter Depth (km)`, `Vs30`, `Joyner-Boore Dist. (km)`, and `PGA (g)` is `-999.0`. In the NGA-West2 database standard, `-999` indicates a missing or unmeasured value. We investigate this in detail in the next section.

# 5. Data Quality Check

A thorough data quality audit requires checking for:
1. Duplicate rows across the entire dataset.
2. Duplicate records based on primary identifiers like `Record Sequence Number`.
3. Sentinel missing values (`-999` and `-999.0`) used in place of standard nulls.
4. Constant columns containing no variance or information.
5. Columns labeled explicitly as unused in the database.
6. Visualizations of missing values, distributions, and boxplots for outliers.

In [ ]:
full_duplicates = raw_df.duplicated().sum()
rsn_duplicates = raw_df["Record Sequence Number"].duplicated().sum()

print("Full duplicate rows:", full_duplicates)
print("Duplicate Record Sequence Numbers:", rsn_duplicates)

We search for occurrences of sentinel codes (`-999`, `-999.0`, and `"-999"`):

In [ ]:
sentinel_count_int = (raw_df == -999).sum().sum()
sentinel_count_float = (raw_df == -999.0).sum().sum()
sentinel_count_str = (raw_df == "-999").sum().sum()

print("Occurrences of integer -999:", sentinel_count_int)
print("Occurrences of float -999.0:", sentinel_count_float)
print("Occurrences of string '-999':", sentinel_count_str)

Next, we scan for constant columns (columns with 1 or fewer unique values) and columns designated as unused:

In [ ]:
constant_cols = [col for col in raw_df.columns if raw_df[col].nunique() <= 1]
print("Constant Columns (no variation):", constant_cols)

unused_cols = [col for col in raw_df.columns if "unused" in col.lower() or "not used" in col.lower()]
print("Unused Columns explicitly labeled:", unused_cols)

To accurately evaluate missing data, we create a working copy `df_working` where sentinel `-999` values are mapped to standard `np.nan`:

In [ ]:
df_working = raw_df.copy()
df_working = df_working.replace(-999, np.nan)
df_working = df_working.replace(-999.0, np.nan)
df_working = df_working.replace("-999", np.nan)
print("Sentinel -999 replacement completed. Total NaNs across dataset:", df_working.isna().sum().sum())

### 5.1 Missing Value Visualization
We plot the true missing value percentages across prominent seismological features:

In [ ]:
audit_features = [
    "Earthquake Magnitude", "Hypocenter Depth (km)", "Strike (deg)", "Dip (deg)", "Rake Angle (deg)",
    "Vs30 (m/s) selected for analysis", "Preferred NEHRP Based on Vs30",
    "EpiD (km)", "HypD (km)", "Joyner-Boore Dist. (km)", "ClstD (km)", "Campbell R Dist. (km)",
    "PGA (g)", "PGV (cm/sec)", "PGD (cm)"
]

missing_percentages = (df_working[audit_features].isna().sum() / len(df_working)) * 100

plt.figure(figsize=(12, 5))
bars = plt.bar(audit_features, missing_percentages, color="#e74c3c", edgecolor="black", alpha=0.85)
plt.title("Missing Value Percentage for Key Earthquake Variables (After -999 Replacement)")
plt.ylabel("Missing Percentage (%)")
plt.xlabel("Variables")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, max(missing_percentages) + 10)
for bar in bars:
    yval = bar.get_height()
    if yval > 0.05:
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.8, f"{yval:.1f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

### 5.2 Distribution Plots for Important Numerical Variables
We visualize the distributions of magnitude, soil shear-wave velocity ($V_{s30}$), closest distance to rupture ($ClstD$), and peak ground acceleration ($PGA$):

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

sns.histplot(df_working["Earthquake Magnitude"].dropna(), bins=30, kde=True, ax=axes[0, 0], color="#2980b9")
axes[0, 0].set_title("Distribution of Earthquake Magnitude")
axes[0, 0].set_xlabel("Magnitude (Mw)")

sns.histplot(df_working["Vs30 (m/s) selected for analysis"].dropna(), bins=30, kde=True, ax=axes[0, 1], color="#27ae60")
axes[0, 1].set_title("Distribution of Vs30 (m/s)")
axes[0, 1].set_xlabel("Vs30 (m/s)")

sns.histplot(df_working["ClstD (km)"].dropna(), bins=30, kde=True, ax=axes[1, 0], color="#8e44ad")
axes[1, 0].set_title("Distribution of Closest Distance (ClstD km)")
axes[1, 0].set_xlabel("Closest Distance (km)")

sns.histplot(df_working["PGA (g)"].dropna(), bins=30, kde=True, ax=axes[1, 1], color="#d35400")
axes[1, 1].set_title("Distribution of Peak Ground Acceleration (PGA g)")
axes[1, 1].set_xlabel("PGA (g)")

plt.tight_layout()
plt.show()

### 5.3 Boxplots for Outlier Diagnosis
Boxplots provide an overview of the spread and extreme observations for these physical variables:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4.5))

sns.boxplot(y=df_working["Earthquake Magnitude"].dropna(), ax=axes[0], color="#aed6f1")
axes[0].set_title("Earthquake Magnitude")
axes[0].set_ylabel("Magnitude (Mw)")

sns.boxplot(y=df_working["Vs30 (m/s) selected for analysis"].dropna(), ax=axes[1], color="#a9dfbf")
axes[1].set_title("Vs30 (m/s)")
axes[1].set_ylabel("Shear-Wave Velocity (m/s)")

sns.boxplot(y=df_working["ClstD (km)"].dropna(), ax=axes[2], color="#d7bde2")
axes[2].set_title("Closest Distance")
axes[2].set_ylabel("Distance (km)")

sns.boxplot(y=df_working["PGA (g)"].dropna(), ax=axes[3], color="#f5b7b1")
axes[3].set_title("PGA")
axes[3].set_ylabel("Acceleration (g)")

plt.tight_layout()
plt.show()

# 6. Identify Important Project Variables

Based strictly on the actual columns present in the NGA-West2 dataset, we categorize candidate attributes into the functional domains required for earthquake ground-motion analysis:

1. **Earthquake Information**: Unique event identifier (`EQID`), name, year, and record sequence number (`Record Sequence Number`).
2. **Site / Station Information**: Station identifier (`Station ID  No.`), station name (`Station Name`).
3. **Magnitude**: Moment magnitude (`Earthquake Magnitude`), magnitude determination type (`Magnitude Type`).
4. **Distance Measures**: Epicentral distance (`EpiD`), hypocentral distance (`HypD`), Joyner-Boore distance (`Joyner-Boore Dist. (km)`), closest distance to ruptured fault plane (`ClstD (km)`), Campbell distance (`Campbell R Dist. (km)`).
5. **Site / Soil Properties**: Time-averaged shear-wave velocity in upper 30m (`Vs30 (m/s) selected for analysis`), NEHRP site classification (`Preferred NEHRP Based on Vs30`).
6. **Ground-Motion Measures**: Peak ground acceleration (`PGA (g)`), peak ground velocity (`PGV (cm/sec)`), peak ground displacement (`PGD (cm)`).
7. **Geographical Information**: Earthquake hypocenter coordinates (`Hypocenter Latitude`, `Hypocenter Longitude`), station coordinates (`Station Latitude`, `Station Longitude`).
8. **Horizontal Recording Information**: Azimuths of horizontal instrument channels (`H1 azimth`, `H2 azimith`).

Below is the structured dictionary table of candidate variables, their physical meaning, data type, and Keep/Drop recommendation:

In [ ]:
variable_dictionary = [
    {"Column": "Record Sequence Number", "Meaning/Role": "Unique ground motion record ID in NGA-West2", "Data Type": "Integer", "Keep/Drop": "Keep", "Reason": "Primary unique key for tracking records"},
    {"Column": "EQID", "Meaning/Role": "Earthquake event ID", "Data Type": "Integer", "Keep/Drop": "Keep", "Reason": "Crucial for grouping stations that recorded the same event"},
    {"Column": "Earthquake Name", "Meaning/Role": "Name of the earthquake event", "Data Type": "String", "Keep/Drop": "Keep", "Reason": "Useful metadata for inspection and reporting"},
    {"Column": "YEAR", "Meaning/Role": "Year of occurrence", "Data Type": "Integer", "Keep/Drop": "Keep", "Reason": "Temporal metadata"},
    {"Column": "Station Name", "Meaning/Role": "Name of the seismic recording station", "Data Type": "String", "Keep/Drop": "Keep", "Reason": "Station identifier metadata"},
    {"Column": "Station Latitude", "Meaning/Role": "Latitude coordinate of recording station", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Required to compute inter-site separation distances"},
    {"Column": "Station Longitude", "Meaning/Role": "Longitude coordinate of recording station", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Required to compute inter-site separation distances"},
    {"Column": "Earthquake Magnitude", "Meaning/Role": "Moment magnitude (Mw) of earthquake", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Primary physical feature governing ground-motion energy"},
    {"Column": "Hypocenter Depth (km)", "Meaning/Role": "Focal depth of earthquake rupture initiation", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Governs path attenuation and wave propagation depth"},
    {"Column": "Hypocenter Latitude (deg)", "Meaning/Role": "Latitude coordinate of earthquake hypocenter", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Source location geometry"},
    {"Column": "Hypocenter Longitude (deg)", "Meaning/Role": "Longitude coordinate of earthquake hypocenter", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Source location geometry"},
    {"Column": "Strike (deg)", "Meaning/Role": "Strike angle of the fault rupture plane", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Fault rupture geometry and orientation"},
    {"Column": "Dip (deg)", "Meaning/Role": "Dip angle of the fault rupture plane", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Fault rupture geometry"},
    {"Column": "Rake Angle (deg)", "Meaning/Role": "Slip direction angle on fault plane", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Defines focal slip kinematics"},
    {"Column": "Mechanism Based on Rake Angle", "Meaning/Role": "Fault mechanism class (strike-slip, normal, reverse)", "Data Type": "Integer/Categorical", "Keep/Drop": "Keep", "Reason": "Standard seismological classification of faulting style"},
    {"Column": "Vs30 (m/s) selected for analysis", "Meaning/Role": "Shear-wave velocity in top 30 meters of soil", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Standard site amplification parameter in engineering"},
    {"Column": "Preferred NEHRP Based on Vs30", "Meaning/Role": "NEHRP soil category (A, B, C, D, E)", "Data Type": "String/Categorical", "Keep/Drop": "Keep", "Reason": "Categorical site condition parameter"},
    {"Column": "EpiD (km)", "Meaning/Role": "Epicentral distance from source to station", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Horizontal source-to-site distance"},
    {"Column": "HypD (km)", "Meaning/Role": "Hypocentral distance from focus to station", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Direct slant distance from source initiation"},
    {"Column": "Joyner-Boore Dist. (km)", "Meaning/Role": "Shortest distance to surface projection of fault", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Standard distance metric in modern GMMs"},
    {"Column": "ClstD (km)", "Meaning/Role": "Closest distance to rupture surface (Rrup)", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Fundamental distance metric in attenuation equations"},
    {"Column": "PGA (g)", "Meaning/Role": "Peak Ground Acceleration (RotD50)", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Primary ground-motion intensity measure"},
    {"Column": "PGV (cm/sec)", "Meaning/Role": "Peak Ground Velocity (RotD50)", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Key velocity-sensitive intensity measure"},
    {"Column": "PGD (cm)", "Meaning/Role": "Peak Ground Displacement (RotD50)", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Displacement-sensitive intensity measure"},
    {"Column": "Campbell R Dist. (km)", "Meaning/Role": "Campbell distance measure", "Data Type": "Float", "Keep/Drop": "Drop", "Reason": "High missing rate (>40%) and redundant with ClstD and Rjb"},
    {"Column": "Unused Column", "Meaning/Role": "Empty placeholder column in database", "Data Type": "Float", "Keep/Drop": "Drop", "Reason": "Contains zero data"},
    {"Column": "Theta.D (deg)", "Meaning/Role": "NGA-West2 directivity angle in degrees", "Data Type": "Float", "Keep/Drop": "Keep", "Reason": "Retained for traceability and downstream proxy analysis; not the paper's maximum-shaking orientation"}
]

variables_table = pd.DataFrame(variable_dictionary)
variables_table

# 7. Initial Feature Selection

From the 274 total columns in the raw dataset, we form an initial feature subset aligned with the reference paper methodology:
- **Earthquake / Source Features**: `Earthquake Magnitude`, `Hypocenter Depth (km)`, `Hypocenter Latitude (deg)`, `Hypocenter Longitude (deg)`, `Strike (deg)`, `Dip (deg)`, `Rake Angle (deg)`, `Mechanism Based on Rake Angle`.
- **Propagation Distances**: `ClstD (km)`, `Joyner-Boore Dist. (km)`, `HypD (km)`, `EpiD (km)`.
- **Site / Soil Characteristics**: `Vs30 (m/s) selected for analysis`, `Preferred NEHRP Based on Vs30`.
- **Ground-Motion Intensity**: `PGA (g)`, `PGV (cm/sec)`, `PGD (cm)`.
- **Station Coordinates & Identifiers**: `Station Latitude`, `Station Longitude`, `Record Sequence Number`, `EQID`, `Earthquake Name`, `YEAR`, `Station Name`.

We drop:
- 111 response spectral acceleration columns (`T0.010S` to `T20.000S`) which are redundant with PGA and PGV at this stage.
- 45 columns with >80% missing data (e.g. `Static Stress Drop`, `Rise Time`, `Fault Rupture Area`).
- Most other directivity and azimuth parameters. We retain `Theta.D (deg)` and `Source to Site Azimuth (deg)` as clearly labeled geometry fields for traceability and downstream proxy analysis; neither is the paper’s waveform-derived maximum-shaking orientation.
- Constant columns (`d`, `Unused Column`, etc.).

Note: Final forward/backward stepwise feature selection requires a calibrated regression model and target variable $\Delta\theta$, which will be performed in subsequent project phases.

In [ ]:
feature_columns = [
    "Record Sequence Number",
    "EQID",
    "Earthquake Name",
    "YEAR",
    "Station Name",
    "Station Latitude",
    "Station Longitude",
    "Earthquake Magnitude",
    "Hypocenter Depth (km)",
    "Hypocenter Latitude (deg)",
    "Hypocenter Longitude (deg)",
    "Strike (deg)",
    "Dip (deg)",
    "Rake Angle (deg)",
    "Mechanism Based on Rake Angle",
    "Vs30 (m/s) selected for analysis",
    "Preferred NEHRP Based on Vs30",
    "EpiD (km)",
    "HypD (km)",
    "Joyner-Boore Dist. (km)",
    "ClstD (km)",
    "PGA (g)",
    "PGV (cm/sec)",
    "PGD (cm)",
    "Theta.D (deg)",
    "Source to Site Azimuth (deg)"
]

df_selected = df_working[feature_columns].copy()
print("Number of Selected Features:", len(feature_columns))
print("Selected DataFrame Shape:", df_selected.shape)

Displaying the first 5 rows of the selected subset:

In [ ]:
df_selected.head()

# 8. Data Cleaning

We apply systematic, explainable cleaning rules based on domain principles and the reference paper:

1. **Step 1: Check and Remove Duplicates**: Ensure no identical rows or duplicate sequence numbers exist.
2. **Step 2: Filter for Engineering Significance ($M_w \ge 5.0$)**: In accordance with Poulos & Silva-Lopez (Section 3.1), ground motions from very small earthquakes ($M_w < 5.0$) do not cause significant structural shaking and are filtered out. This also removes unmeasured magnitude records.
3. **Step 3: Require Valid Ground-Motion Intensities**: Remove records missing `PGA (g)`, `PGV (cm/sec)`, or `PGD (cm)`.
4. **Step 4: Require Valid Site and Distance Attributes**: Remove records missing primary distance metrics (`ClstD (km)`), hypocenter depth, or site velocity (`Vs30`).
5. **Step 5: Require Flatfile Geometry Fields**: Retain rows with `Theta.D (deg)` and source-to-site azimuth values. These are directivity/geometry descriptors, not the paper’s waveform-derived orientation target.

For every cleaning step, we show the exact row counts before and after the operation.

In [ ]:
count_initial = len(df_selected)
print("Before cleaning:", count_initial, "rows")

df_step1 = df_selected.drop_duplicates()
count_step1 = len(df_step1)
print("After removing duplicate rows:", count_step1, "rows (Removed:", count_initial - count_step1, ")")

df_step2 = df_step1[df_step1["Earthquake Magnitude"] >= 5.0]
count_step2 = len(df_step2)
print("After filtering for Magnitude >= 5.0:", count_step2, "rows (Removed:", count_step1 - count_step2, ")")

df_step3 = df_step2.dropna(subset=["PGA (g)", "PGV (cm/sec)", "PGD (cm)"])
count_step3 = len(df_step3)
print("After requiring valid ground-motion intensities (PGA, PGV, PGD):", count_step3, "rows (Removed:", count_step2 - count_step3, ")")

df_step4 = df_step3.dropna(subset=["ClstD (km)", "Vs30 (m/s) selected for analysis", "Hypocenter Depth (km)"])
count_step4 = len(df_step4)
print("After requiring valid distance, depth, and Vs30:", count_step4, "rows (Removed:", count_step3 - count_step4, ")")

df_step5 = df_step4.dropna(subset=["Theta.D (deg)", "Source to Site Azimuth (deg)"])
count_step5 = len(df_step5)
print("After requiring Theta.D and source-to-site azimuth fields:", count_step5,
      "rows (Removed:", count_step4 - count_step5, ")")


Summary of cleaning progression for the supplied workbook:
- Before cleaning: 21,540 records.
- After duplicate removal: 21,540 records (no exact duplicate rows).
- After requiring magnitude >= 5.0: 8,495 records (13,045 removed).
- After requiring PGA, PGV, and PGD: 8,409 records (86 removed).
- After requiring distance, depth, and Vs30: 8,391 records (18 removed).
- After requiring `Theta.D` and source-to-site azimuth: 7,486 records (905 removed).
- Final cleaning output: 7,486 records and 26 selected columns.

`Theta.D` is retained as a flatfile directivity field for reproducibility. It is not the waveform-derived angle of maximum ground shaking used by the reference paper.

# 9. Missing Value Treatment

We evaluate the remaining missing values across our selected features to determine whether any imputation is required.

In [ ]:
remaining_missing = df_step5.isna().sum()
missing_percent = (remaining_missing / len(df_step5)) * 100

missing_treatment_df = pd.DataFrame({
    "Feature": df_step5.columns,
    "Missing Count": remaining_missing.values,
    "Missing Percentage (%)": missing_percent.values
})

missing_treatment_df

All 26 selected features have **0 missing values** after applying our domain-based cleaning rules. Every retained record has complete, reliable measurements for all source, path, site, and intensity features. No synthetic data imputation (which could distort earthquake wave physics) is necessary.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(["Initial Raw Data", "Cleaned Data"], [sentinel_count_int, df_step5.isna().sum().sum()], color=["#e74c3c", "#27ae60"], edgecolor="black")
ax.set_title("Total Missing / Sentinel Values: Raw vs Cleaned Dataset")
ax.set_ylabel("Count of Missing / Sentinel Values")
for i, v in enumerate([sentinel_count_int, df_step5.isna().sum().sum()]):
    ax.text(i, v + 25000, f"{v:,}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

# 10. Categorical Variables

We distinguish between identifier metadata columns and generalizable physical categorical variables:
- **Identifiers (High Cardinality)**: `EQID`, `Earthquake Name`, `Station Name`. These are labels and should NOT be one-hot encoded as predictive ML features.
- **Physical Categorical Features**:
  1. `Preferred NEHRP Based on Vs30`: Soil condition classification (classes A, B, C, D, E).
  2. `Mechanism Based on Rake Angle`: Seismological fault mechanism (0 = Strike-slip, 1 = Normal, 2 = Reverse, 3 = Reverse-oblique, 4 = Normal-oblique).

Below, we inspect the unique categories and show standard one-hot encoding on the site classification variable.

In [ ]:
print("NEHRP Soil Classification Distribution:")
print(df_step5["Preferred NEHRP Based on Vs30"].value_counts())

print("\nFault Mechanism Distribution:")
print(df_step5["Mechanism Based on Rake Angle"].value_counts())

Standardizing hybrid NEHRP classes (e.g. grouping border classes `D/E` and `D-E` into `D`, `C/D` into `C`) and demonstrating one-hot encoding:

In [ ]:
nehrp_clean = df_step5["Preferred NEHRP Based on Vs30"].astype(str).str.strip().replace({
    "D/E": "D",
    "D-E": "D",
    "C/D": "C"
})

nehrp_dummies = pd.get_dummies(nehrp_clean, prefix="NEHRP", drop_first=True, dtype=int)
nehrp_dummies.head()

# 11. Numerical Feature Preparation

We evaluate the numerical properties (minimum, maximum, mean, and standard deviation) of the features intended for machine learning.

**Crucial Machine Learning Principle**:
Feature scaling (such as StandardScaler or MinMaxScaler) must **NOT** be applied to the entire dataset at this stage. Doing so would cause **Data Leakage**, as test-set statistics would influence the training transformation. Scaling should be fitted strictly on the training partition after the train/test split in the modeling phase.

In [ ]:
ml_numeric_features = [
    "Earthquake Magnitude",
    "Hypocenter Depth (km)",
    "Strike (deg)",
    "Dip (deg)",
    "Rake Angle (deg)",
    "Vs30 (m/s) selected for analysis",
    "EpiD (km)",
    "HypD (km)",
    "Joyner-Boore Dist. (km)",
    "ClstD (km)",
    "PGA (g)",
    "PGV (cm/sec)",
    "PGD (cm)"
]

numeric_stats_table = pd.DataFrame({
    "Feature": ml_numeric_features,
    "Min": df_step5[ml_numeric_features].min().values,
    "Max": df_step5[ml_numeric_features].max().values,
    "Mean": df_step5[ml_numeric_features].mean().values,
    "Std Dev": df_step5[ml_numeric_features].std().values
})

numeric_stats_table

### Descriptive statistics for flatfile geometry angles

These are summaries of supplied geometry fields, not the paper’s station-specific maximum-shaking orientation.

In [ ]:
angle_geometry_stats = df_step5[["Theta.D (deg)", "Source to Site Azimuth (deg)"]].describe().T
angle_geometry_stats


# 12. Outlier Analysis

In earthquake datasets, extreme observations are naturally occurring physical phenomena rather than sensor errors:
- Very high accelerations ($PGA > 1.0\text{ g}$) represent rare, destructive near-fault shockwaves (e.g. Tarzana station in the 1994 Northridge earthquake).
- Very large magnitudes ($M_w > 7.5$) represent major earthquakes (e.g. Chi-Chi, Hector Mine, Tohoku).
- High shear wave velocities ($Vs30 > 1500\text{ m/s}$) reflect hard crystalline bedrock sites.

Blindly deleting statistical outliers using arbitrary rules like $3\sigma$ or $1.5 \times IQR$ would discard the most critical, high-energy earthquake observations that ML models must learn. Therefore, records are retained because all values fall within physically valid geophysical boundaries.

In [ ]:
pga_extreme = df_step5[df_step5["PGA (g)"] > 1.0]
print("Count of extreme PGA records (> 1.0 g):", len(pga_extreme))
print("Earthquakes with PGA > 1.0 g:", pga_extreme["Earthquake Name"].unique().tolist())

mag_extreme = df_step5[df_step5["Earthquake Magnitude"] >= 7.5]
print("Count of major earthquake records (Mw >= 7.5):", len(mag_extreme))
print("Major earthquakes:", mag_extreme["Earthquake Name"].unique().tolist())

Visualizing the distribution of PGA on standard and logarithmic scales to demonstrate that extreme values follow a natural log-normal decay:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(df_step5["PGA (g)"], bins=40, kde=True, ax=axes[0], color="#e67e22")
axes[0].set_title("PGA (Linear Scale)")
axes[0].set_xlabel("PGA (g)")

sns.histplot(np.log10(df_step5["PGA (g)"]), bins=40, kde=True, ax=axes[1], color="#2980b9")
axes[1].set_title("PGA (Log10 Scale - Natural Physical Form)")
axes[1].set_xlabel("log10(PGA)")

plt.tight_layout()
plt.show()

# 13. Final Cleaned Dataset

We assign the cleaned dataset to the final variable name `cleaned_df`, reset the index, and verify its structure.

In [ ]:
cleaned_df = df_step5.copy().reset_index(drop=True)

print("Cleaned DataFrame Shape:", cleaned_df.shape)
print("Remaining Missing Values Count:", cleaned_df.isna().sum().sum())
print("Duplicate Rows Count:", cleaned_df.duplicated().sum())
print("Columns Present in cleaned_df:")
print(cleaned_df.columns.tolist())

Displaying the first 5 records of `cleaned_df`:

In [ ]:
cleaned_df.head()

# 14. Final Data Quality Report

A compact comparison summarizing the transformation from the raw Excel flatfile to the cleaned dataset:

In [ ]:
quality_report = pd.DataFrame({
    "Metric": [
        "Original Rows",
        "Final Cleaned Rows",
        "Original Columns",
        "Final Cleaned Columns",
        "Duplicate Rows Removed",
        "Non-Significant / Incomplete Rows Removed",
        "Columns Removed (Spectral / High Missing / Constant)",
        "Missing Values Remaining",
        "Important Features Retained"
    ],
    "Value": [
        raw_df.shape[0],
        cleaned_df.shape[0],
        raw_df.shape[1],
        cleaned_df.shape[1],
        0,
        raw_df.shape[0] - cleaned_df.shape[0],
        raw_df.shape[1] - cleaned_df.shape[1],
        cleaned_df.isna().sum().sum(),
        len(cleaned_df.columns)
    ]
})

quality_report

# 15. Final Visualizations

We generate key project-specific visualizations to illustrate fundamental physical relationships:
1. Distance distributions ($ClstD$, $HypD$, $Joyner-Boore$).
2. Peak Ground Motion distributions ($PGA$, $PGV$).
3. Soil velocity ($Vs30$) distribution by NEHRP site class.
4. Physical relationships: Magnitude vs PGA and Distance vs PGA (attenuation curve).
5. Correlation heatmap across selected continuous seismological variables.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

sns.histplot(cleaned_df["ClstD (km)"], bins=30, kde=True, ax=axes[0], color="#3498db")
axes[0].set_title("Closest Distance (ClstD km)")
axes[0].set_xlabel("ClstD (km)")

sns.histplot(cleaned_df["HypD (km)"], bins=30, kde=True, ax=axes[1], color="#9b59b6")
axes[1].set_title("Hypocentral Distance (HypD km)")
axes[1].set_xlabel("HypD (km)")

sns.histplot(cleaned_df["Joyner-Boore Dist. (km)"], bins=30, kde=True, ax=axes[2], color="#1abc9c")
axes[2].set_title("Joyner-Boore Distance (Rjb km)")
axes[2].set_xlabel("Rjb (km)")

plt.tight_layout()
plt.show()

Visualizing PGA and PGV distributions:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(cleaned_df["PGA (g)"], bins=35, kde=True, ax=axes[0], color="#e74c3c")
axes[0].set_title("Peak Ground Acceleration (PGA g)")
axes[0].set_xlabel("PGA (g)")

sns.histplot(cleaned_df["PGV (cm/sec)"], bins=35, kde=True, ax=axes[1], color="#e67e22")
axes[1].set_title("Peak Ground Velocity (PGV cm/s)")
axes[1].set_xlabel("PGV (cm/sec)")

plt.tight_layout()
plt.show()

Visualizing $Vs30$ distribution across NEHRP site classes:

In [ ]:
valid_nehrp = cleaned_df[cleaned_df["Preferred NEHRP Based on Vs30"].isin(["A", "B", "C", "D", "E"])]

plt.figure(figsize=(9, 5))
sns.boxplot(x="Preferred NEHRP Based on Vs30", y="Vs30 (m/s) selected for analysis", data=valid_nehrp, order=["A", "B", "C", "D", "E"], palette="Set2")
plt.title("Vs30 Distribution across NEHRP Site Classes")
plt.xlabel("NEHRP Site Class")
plt.ylabel("Vs30 (m/s)")
plt.tight_layout()
plt.show()

Physical relationships: Magnitude vs PGA and Distance vs PGA (demonstrating attenuation with distance):

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(x="Earthquake Magnitude", y="PGA (g)", data=cleaned_df, alpha=0.3, color="#2c3e50", ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Earthquake Magnitude vs PGA (RotD50)")
axes[0].set_xlabel("Moment Magnitude (Mw)")
axes[0].set_ylabel("PGA (g) [Log Scale]")

sns.scatterplot(x="ClstD (km)", y="PGA (g)", data=cleaned_df, alpha=0.3, color="#c0392b", ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_xscale("log")
axes[1].set_title("Ground-Motion Attenuation: Distance vs PGA")
axes[1].set_xlabel("Closest Distance (km) [Log Scale]")
axes[1].set_ylabel("PGA (g) [Log Scale]")

plt.tight_layout()
plt.show()

Correlation heatmap across selected numerical earthquake features:

In [ ]:
corr_features = [
    "Earthquake Magnitude",
    "Hypocenter Depth (km)",
    "Vs30 (m/s) selected for analysis",
    "ClstD (km)",
    "Joyner-Boore Dist. (km)",
    "PGA (g)",
    "PGV (cm/sec)",
    "PGD (cm)"
]

corr_matrix = cleaned_df[corr_features].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title("Correlation Matrix of Key Seismological Variables")
plt.tight_layout()
plt.show()

### Flatfile angle diagnostics

These plots describe the retained flatfile geometry fields only. `Theta.D` is a directivity angle; it is not the waveform-derived maximum-shaking orientation or a computed station-pair $\Delta\theta$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(cleaned_df["Theta.D (deg)"], bins=30, ax=axes[0], color="#287c8e")
axes[0].set_title("Flatfile directivity angle Theta.D")
axes[0].set_xlabel("Theta.D (degrees)")

sns.histplot(cleaned_df["Source to Site Azimuth (deg)"], bins=36, ax=axes[1], color="#e07a3f")
axes[1].set_title("Source-to-site azimuth")
axes[1].set_xlabel("Azimuth (degrees)")

plt.tight_layout()
plt.show()


# 16. Save the Cleaned Dataset

We save `cleaned_df` to a CSV file named `cleaned_dataset_NGA_West2_d050.csv` in the `dataset` folder. The original Excel file remains completely untouched.

In [ ]:
output_filename = "dataset/cleaned_dataset_NGA_West2_d050.csv"
cleaned_df.to_csv(output_filename, index=False)
print("Cleaned dataset successfully saved to:", output_filename)

Verifying that the exported file exists and checking its size and content:

In [ ]:
verify_df = pd.read_csv(output_filename)
print("Verification Shape:", verify_df.shape)
print("Verification Missing Values:", verify_df.isna().sum().sum())

# 17. Final Summary

### 1. What Was Cleaned and Diagnostic Findings
- **Sentinel Missing Value Replacement**: In NGA-West2 flatfiles, missing or unmeasured quantities are encoded with the sentinel code `-999` rather than standard blank cells. We mapped all instances of `-999` to `NaN` so that true missingness could be analyzed.
- **Deduplication**: Verified that no duplicate rows or duplicate `Record Sequence Numbers` exist.
- **Engineering Significance Filtering**: Applied the domain filter established in the reference paper (Poulos & Silva-Lopez, 2020), retaining records with $M_w \ge 5.0$. This filtered out 13,045 non-damaging micro-earthquake records and invalid magnitude entries.
- **Completeness Filtering**: Removed records missing ground-motion intensities (`PGA`, `PGV`, `PGD`) or primary distance/site measurements (`ClstD`, `Vs30`, `Hypocenter Depth`).
- **Feature Pruning**: Excluded 111 response spectra period columns (`T0.010S` to `T20.000S`), 45 columns with >80% missing data, constant columns, and unassigned placeholder columns.

### 2. Retained Features and Data Dimensions
- **Final Rows**: 7,486 records with complete selected source, path, site, motion, and flatfile geometry fields.
- **Final Columns**: 26 selected fields covering earthquake source properties, propagation distances, site soil parameters, station coordinates, and ground-motion intensities.
- **Data Integrity**: Exactly 0 missing values remain across all retained features.
- **Outlier Handling**: Extreme values (such as $PGA > 1.0\text{ g}$ or $M_w \ge 7.5$) were verified as genuine physical earthquake observations and retained.

### 3. Angle fields and the later pairwise proxy
- **`Theta.D (deg)`** is retained as the NGA-West2 flatfile directivity angle. It is not the RotD100 direction of maximum shaking described in the paper.
- The supplied RotD50 flatfile does not contain the acceleration waveforms needed to calculate the paper’s station-specific maximum-shaking angles.
- The downstream repository script currently subtracts nearby `Theta.D` values as a **proxy target**. It is reproducible, but it is not the paper’s true waveform-derived $\Delta\theta$ and its scale should not be compared directly with the paper.
- This data-cleaning notebook prepares the single-record table only; it does not calculate station pairs or $\Delta\theta$.

### 4. Next Steps for Team Members
With `dataset/cleaned_dataset_NGA_West2_d050.csv` ready, the next phases of the project are:
1. **Station Pairing**: The current downstream script groups records by `EQID` and pairs stations from the same earthquake. It uses differences in `Theta.D` only as a proxy, not as the paper's waveform-derived response.
2. **Spatial Separation Distance ($d$)**: Compute inter-station distances using station latitudes and longitudes (e.g. limiting to $d \le 5\text{ km}$ as in the reference paper).
3. **Model Training & Evaluation**: Evaluate any model against the clearly labeled proxy target; do not interpret it as the paper's true maximum-shaking directionality without waveform-derived angles.